In [46]:


import polars as pl

In [47]:
K_TOP = 50

In [81]:
def preprocess(document: pl.DataFrame) -> pl.DataFrame:
    return (document
            .filter(pl.col("lang") == "ENGLISH")
            .select(pl.col('id'), pl.col('text').str.split(' ').alias('term'))
            .explode('term')
            .filter(pl.col("term").str.contains(r'\d|\.|"').not_())
            .group_by('id', 'term')
            .len())


def fingerprint(document: pl.DataFrame) -> pl.DataFrame:
    total = document.sum()['len'][0]

    return (
        document
        .group_by('term')
        .agg(pl.col('len').sum())
        .with_columns(freq=pl.col('len') / total)
        .drop('len')
        .top_k(k=K_TOP, by='freq')
        .sort('freq', descending=True)
    )


def fingerprint_file(name: str) -> pl.DataFrame:
    return fingerprint(preprocess(pl.read_parquet(name)))


def compare(fp1: pl.DataFrame, fp2: pl.DataFrame) -> float:
    return (fp1
    .join(fp2, left_on='term', right_on='term')
    .with_columns(diff=pow(abs(pl.col('freq') - pl.col('freq_right')), pl.col('freq')))
    .drop('freq', 'freq_right', 'term')
    .sum()['diff'][0])


df = pl.read_parquet('/Users/qr/Downloads/v5_chunks/c00431.parquet')



In [88]:
(df
 .group_by('domain')
 .len()
 .top_k(k=3, by='len')
 .join(df, left_on='domain', right_on='domain'))

domain,len,id,url,timestamp,duration,http_version,status,header,type,lang,lang_confidence,text,outflow
i64,u32,i64,str,i64,i64,i32,i32,str,i32,str,f32,str,str
3774953,179,810746688,"""https://media.dsp-wiki.com/b/b…",1769533128000,198,2,200,"""{x-amz-meta-s3cmd-attrs=[atime…",5,"""UNKNOWN""",0.0,"""""","""{}"""
3774953,179,810746696,"""https://media.dsp-wiki.com/5/5…",1769533185533,216,2,200,"""{x-amz-meta-s3cmd-attrs=[atime…",5,"""UNKNOWN""",0.0,"""""","""{}"""
3774953,179,810746710,"""https://media.dsp-wiki.com/d/d…",1769533190015,94,2,200,"""{x-amz-meta-s3cmd-attrs=[atime…",5,"""UNKNOWN""",0.0,"""""","""{}"""
3774953,179,810746727,"""https://media.dsp-wiki.com/7/7…",1769533272956,182,2,200,"""{x-amz-meta-s3cmd-attrs=[atime…",5,"""UNKNOWN""",0.0,"""""","""{}"""
3774953,179,810746736,"""https://media.dsp-wiki.com/b/b…",1769533347873,173,2,200,"""{x-amz-meta-s3cmd-attrs=[atime…",5,"""UNKNOWN""",0.0,"""""","""{}"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
5598711,202,810901707,"""http://spheresofpower.wikidot.…",1770661460681,192,2,200,"""{""date"":[""Mon, 09 Feb 2026 18:…",2,"""ENGLISH""",0.419942,"""wraith sphere power wiki wikid…","""{}"""
5598711,202,810901708,"""http://spheresofpower.wikidot.…",1770682166504,161,2,200,"""{""date"":[""Tue, 10 Feb 2026 00:…",2,"""ENGLISH""",0.398315,"""sphere power sphere power wiki…","""{}"""
5598711,202,810901711,"""http://spheresofpower.wikidot.…",1770682191324,174,2,200,"""{""date"":[""Tue, 10 Feb 2026 00:…",2,"""ENGLISH""",0.415496,"""shield sphere power wiki wikid…","""{}"""
